# FactIndic Dual-Archetype Batch Re-Evaluation (Colab T4)

Re-evaluates **both archetypes** (`Roman-script Hinglish` and `Devanagari / Code-Mixed`) across all 4 conditions (`Qwen3 1.7B` & `4B` × `Standard` & `Factuality-aware`, 240 outputs total) using the upgraded guarded pipeline:
- Pure greedy decoding (`do_sample=False, num_beams=1`, no `repetition_penalty` or `no_repeat_ngram_size`)
- Pre-generation **Fact Anchors** (`extract_fact_anchors`: dates, numbers, Roman & Devanagari warning/negation cues)
- Post-generation **Name/Date Guards** + **Verbatim Quote Claim Checker**
- Secondary **`mDeBERTa-v3-base-mnli-xnli`** NLI (direct for Devanagari/code-mixed; gloss-normalized + `0.85` threshold for Roman-script Hinglish)

Outputs `factindic_dual_archetype_reeval_summary.json` and `factindic_dual_archetype_reeval_rows.csv` with paired 95% bootstrap confidence intervals (5,000 resamples, seed 42).

In [ ]:
%pip -q install "transformers>=4.51.3,<5" accelerate sentencepiece pyarrow datasets ipywidgets "git+https://github.com/Sirohi0/factindic.git"

## 1. Load Helpers & Guarded Pipeline

In [ ]:
import ast, difflib, gc, html, io, json, random, re, time, unicodedata
from collections import Counter
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import requests
import torch
from datasets import load_dataset
from IPython.display import display
from google.colab import files
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

from factindic.codemix import code_mix_text

assert torch.cuda.is_available(), "Choose Runtime -> Change runtime type -> T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0))

MODEL_IDS = {
    "Qwen3 1.7B": "Qwen/Qwen3-1.7B",
    "Qwen3 4B": "Qwen/Qwen3-4B",
}
PROMPT_STYLES = ["Standard", "Factuality-aware"]
NLI_MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
NLI_CONTRADICTION_THRESHOLD = 0.85
MAX_SOURCE_TOKENS = 2048
SUMMARY_TOKEN_LIMIT = 192
CLAIM_CHECK_TOKEN_LIMIT = 768
SEED = 42
N_SAMPLE = 30

_loaded_model_key = None
_model = None
_tokenizer = None
_nli_model = None
_nli_tokenizer = None

def load_model(model_key):
    global _loaded_model_key, _model, _tokenizer
    if _loaded_model_key == model_key and _model is not None:
        return _model, _tokenizer
    if _model is not None:
        del _model, _tokenizer
        _model, _tokenizer = None, None
        gc.collect()
        torch.cuda.empty_cache()
    model_id = MODEL_IDS[model_key]
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto").eval()
    _model.generation_config.do_sample = False
    _model.generation_config.temperature = 1.0
    _model.generation_config.top_p = 1.0
    _model.generation_config.top_k = 50
    _loaded_model_key = model_key
    return _model, _tokenizer

def load_nli_model():
    global _nli_model, _nli_tokenizer
    if _nli_model is not None and _nli_tokenizer is not None:
        return _nli_model, _nli_tokenizer
    _nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_ID)
    _nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_ID)
    _nli_model = _nli_model.to("cuda" if torch.cuda.is_available() else "cpu").eval()
    return _nli_model, _nli_tokenizer

_HINGLISH_NLI_GLOSSES = (
    (r"\bbilkul na jaayein\b", "do not go at all"),
    (r"\bna jaayein\b", "do not go"),
    (r"\bdoor rehne ko kaha(?: gaya hai)?\b", "told to stay away"),
    (r"\bdoor rehne\b", "stay away"),
    (r"\bkhali kar dein\b", "evacuate immediately"),
    (r"\bband hai\b", "is closed"),
    (r"\balert jaari kiya hai\b", "issued an alert"),
    (r"\bkaafi suhana tha\b", "was quite pleasant"),
    (r"\bthandi hawa\b", "cool breeze"),
    (r"\bkoi bada jam nahi laga\b", "no major traffic jam occurred"),
    (r"\bnahi\b", "not"),
)

def normalize_hinglish_for_nli(text):
    normalized = " ".join(str(text).split())
    for pattern, replacement in _HINGLISH_NLI_GLOSSES:
        normalized = re.sub(pattern, replacement, normalized, flags=re.I)
    return normalized

def nli_check_claim(premise, hypothesis, archetype="roman_hinglish"):
    if not premise or not hypothesis:
        return "unavailable", None
    model, tokenizer = load_nli_model()
    nli_premise = premise if archetype == "devanagari_codemixed" else normalize_hinglish_for_nli(premise)
    encoded = tokenizer(nli_premise, hypothesis, truncation=True, max_length=512, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        logits = model(**encoded).logits[0]
        probs = torch.softmax(logits, dim=-1)
        best_idx = int(torch.argmax(probs).item())
        best_prob = float(probs[best_idx].item())
    id2label = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
    label = id2label.get(best_idx, "unknown")
    if archetype != "devanagari_codemixed" and label == "contradiction" and best_prob < NLI_CONTRADICTION_THRESHOLD:
        label = "uncertain_nli"
    return label, round(best_prob, 3)

def chat_generate(model, tokenizer, user_prompt, max_new_tokens):
    messages = [{"role": "user", "content": user_prompt}]
    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", enable_thinking=False
    ).to(model.device)
    input_tokens = int(encoded["input_ids"].shape[-1])
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **encoded, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
    elapsed = time.perf_counter() - started
    new_ids = output_ids[0][input_tokens:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip(), int(len(new_ids)), elapsed

def prepare_source(source, tokenizer):
    source_ids = tokenizer.encode(source, add_special_tokens=False)
    truncated = len(source_ids) > MAX_SOURCE_TOKENS
    source_used = tokenizer.decode(source_ids[:MAX_SOURCE_TOKENS], skip_special_tokens=True)
    return source_used, len(source_ids), truncated

def normalize_quote(text):
    return " ".join(unicodedata.normalize("NFKC", str(text)).split()).casefold()

def numeric_tokens(text):
    normalized = unicodedata.normalize("NFKC", str(text))
    return re.findall(r"(?<![\w])\d+(?:[.,]\d+)*(?![\w])", normalized)

def unsupported_numeric_tokens(source_text, summary):
    source_values = set(numeric_tokens(source_text))
    return sorted(set(v for v in numeric_tokens(summary) if v not in source_values))

_MONTH_ALIASES = {
    "jan": 1, "january": 1, "feb": 2, "february": 2, "mar": 3, "march": 3,
    "apr": 4, "april": 4, "may": 5, "jun": 6, "june": 6, "jul": 7,
    "july": 7, "aug": 8, "august": 8, "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10, "nov": 11, "november": 11, "dec": 12, "december": 12,
}
_DATE_PATTERN = re.compile(
    r"\b(?:(?P<day>\d{1,2})\s+(?P<month>January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+(?P<year>\d{2,4})|"
    r"(?P<month_first>January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+(?P<day_first>\d{1,2})(?:st|nd|rd|th)?[,]?\s+(?P<year_first>\d{2,4}))\b",
    flags=re.I,
)

def _date_parts(match):
    day = int(match.group("day") or match.group("day_first"))
    month_text = match.group("month") or match.group("month_first")
    year = int(match.group("year") or match.group("year_first"))
    return day, _MONTH_ALIASES[month_text.casefold()], year

def _identifier_number(suffix):
    if not suffix: return None
    if suffix.isdigit(): return int(suffix)
    values = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    total, prev = 0, 0
    for ch in reversed(suffix.upper()):
        val = values.get(ch)
        if val is None: return None
        total += -val if val < prev else val
        prev = max(prev, val)
    return total

def _entity_parts(token):
    m = re.fullmatch(r"(?P<base>[A-Za-z][A-Za-z0-9]*?)(?:[-–](?P<suffix>[0-9]+|[IVXLCDM]+))?", token)
    return (m.group("base").casefold(), _identifier_number(m.group("suffix"))) if m else (token.casefold(), None)

def correct_source_name_spellings(source_text, summary):
    token_pattern = re.compile(r"\b[A-Z][A-Za-z0-9]*(?:[-–](?:[0-9]+|[IVXLCDM]+))?\b")
    source_candidates = []
    for match in token_pattern.finditer(source_text):
        prefix = source_text[:match.start()]
        boundary = max(prefix.rfind(mark) for mark in (".", "!", "?", "।"))
        if prefix[boundary + 1:].strip() or re.search(r"[-–](?:[0-9]+|[IVXLCDM]+)$", match.group(0), re.I):
            source_candidates.append(match.group(0))
    corrections, corrected, seen = [], summary, set()
    for match in list(token_pattern.finditer(summary)):
        gen = match.group(0)
        if gen.casefold() in seen: continue
        seen.add(gen.casefold())
        g_base, g_num = _entity_parts(gen)
        if len(g_base) < 6: continue
        cands = []
        for s_name in source_candidates:
            s_base, s_num = _entity_parts(s_name)
            if s_num != g_num: continue
            score = difflib.SequenceMatcher(None, g_base, s_base).ratio()
            if score >= 0.90: cands.append((score, s_name))
        cands.sort(reverse=True)
        if not cands or (len(cands) > 1 and cands[0][0] - cands[1][0] < 0.06): continue
        best_score, s_name = cands[0]
        if gen.casefold() == s_name.casefold(): continue
        corrected = re.sub(rf"(?<![A-Za-z0-9]){re.escape(gen)}(?![A-Za-z0-9])", lambda _: s_name, corrected, flags=re.I)
        corrections.append({"generated_name": gen, "source_name": s_name, "similarity": round(best_score, 3)})
    return corrected, corrections

def correct_unique_source_date(source_text, summary):
    s_dates, g_dates = list(_DATE_PATTERN.finditer(source_text)), list(_DATE_PATTERN.finditer(summary))
    if len(s_dates) != 1 or len(g_dates) != 1: return summary, None
    sm, gm = s_dates[0], g_dates[0]
    if _date_parts(sm) == _date_parts(gm): return summary, None
    return summary[:gm.start()] + sm.group(0) + summary[gm.end():], {"generated_date": gm.group(0), "source_date": sm.group(0)}

def unsupported_full_dates(source_text, summary):
    s_dates = {_date_parts(m) for m in _DATE_PATTERN.finditer(source_text)}
    return sorted({m.group(0) for m in _DATE_PATTERN.finditer(summary) if _date_parts(m) not in s_dates})

def extract_fact_anchors(source_text, archetype="roman_hinglish"):
    dates = [m.group(0) for m in _DATE_PATTERN.finditer(source_text)]
    numbers = numeric_tokens(source_text)
    cue_patterns = (
        r"bilkul na", r"na jaayein", r"door rehne", r"band hai", r"khali kar",
        r"\bmat\b", r"\bnahi\b", r"\bmana\b", r"\balert\b", r"\bwarning\b",
        r"नहीं", r"मत", r"बंद है", r"चेतावनी", r"अलर्ट", r"दूर रहें", r"खाली कर",
    )
    cues = []
    for p in cue_patterns:
        m = re.search(p, source_text, flags=re.I)
        if m:
            cues.append(m.group(0))
    return {"dates": dates, "numbers": numbers, "cues": cues, "archetype": archetype}

def make_summary(source_used, prompt_style, model, tokenizer, numeric_retry=False, fact_anchors=None, archetype="roman_hinglish"):
    source_desc = "Devanagari Hindi / code-mixed Hindi-English" if archetype == "devanagari_codemixed" else "Roman-script Hinglish"
    shared = (
        f"Translate and summarize the meaning of this {source_desc} source into a concise English "
        "summary of at most two sentences in natural English. Use past tense for completed events. "
        "Keep established names and technical terms exactly as spelled in the source; do not "
        "transliterate numbers into Roman numerals or invent suffixes. Keep the main event and the exact people, places, "
        "numbers, dates, attribution, and outcome stated in the source. Preserve negation, "
        "warnings, uncertainty, and directions exactly: do not reverse whether people should "
        "approach, avoid, leave, or remain in a place. Do not add advice, causes, identities, "
        "or other details. If a detail is unclear, leave it out. Return only the summary."
    )
    instruction = (shared + " Before answering, check that every clause is directly supported by the source and that no translation changed who did what or what people were told.") if prompt_style == "Factuality-aware" else shared
    if numeric_retry:
        instruction += " The previous draft changed a name, date, or number. For this retry, copy every digit sequence and date exactly as written in the source."
    anchors = fact_anchors if fact_anchors is not None else extract_fact_anchors(source_used, archetype=archetype)
    anchor_lines = []
    if anchors.get("dates"): anchor_lines.append("Dates: " + ", ".join(anchors["dates"]))
    if anchors.get("numbers"): anchor_lines.append("Numbers: " + ", ".join(anchors["numbers"]))
    if anchors.get("cues"): anchor_lines.append("Warning/negation cues: " + ", ".join(anchors["cues"]))
    anchor_block = ("\nVerified source anchors (preserve exactly):\n" + "\n".join(anchor_lines)) if anchor_lines else ""
    prompt = f"{instruction}{anchor_block}\n\nSource text:\n{source_used}\n\nEnglish summary:"
    summary, tokens, seconds = chat_generate(model, tokenizer, prompt, SUMMARY_TOKEN_LIMIT)
    return summary, tokens, seconds, tokens >= SUMMARY_TOKEN_LIMIT

def parse_json_object(text):
    text = str(text).strip()
    start = text.find("{")
    if start < 0: raise ValueError("No JSON object")
    stack, quote, escaped, end = [], None, False, None
    pairs = {"}": "{", "]": "["}
    for i in range(start, len(text)):
        ch = text[i]
        if quote:
            if escaped: escaped = False
            elif ch == "\\": escaped = True
            elif ch == quote: quote = None
            continue
        if ch in {'"', "'"}: quote = ch
        elif ch in "{[ ": stack.append(ch)
        elif ch in "}]":
            if not stack or stack[-1] != pairs[ch]: break
            stack.pop()
            if not stack: end = i + 1; break
    cand = text[start:end] if end else text[start:]
    for attempt in (cand, re.sub(r",\s*([}\]])", r"\1", cand)):
        try: return json.loads(attempt)
        except Exception: pass
    try: return ast.literal_eval(re.sub(r",\s*([}\]])", r"\1", cand))
    except Exception: pass
    quoted = re.sub(r"([{,]\s*)([A-Za-z_][A-Za-z0-9_]*)\s*:", r'\1"\2":', re.sub(r",\s*([}\]])", r"\1", cand))
    try: return json.loads(quoted)
    except Exception: return ast.literal_eval(quoted)

def check_claims(source_used, summary, model, tokenizer):
    prompt = (
        "You are checking whether a summary is supported by its source. Split it into up to "
        "12 short, atomic factual claims. For each claim choose exactly one status: supported, contradicted, not_found, or unclear. "
        "For supported or contradicted claims, give one short evidence quote copied verbatim from the source. "
        'Return JSON only: {"claims":[{"claim":"...","status":"supported|contradicted|not_found|unclear","evidence":"exact source quote or empty"}]}\n\n'
        f"SOURCE:\n{source_used}\n\nSUMMARY:\n{summary}\n"
    )
    raw, tokens, seconds = chat_generate(model, tokenizer, prompt, CLAIM_CHECK_TOKEN_LIMIT)
    try:
        parsed = parse_json_object(raw)
        if not isinstance(parsed, dict) or not isinstance(parsed.get("claims"), list): raise ValueError("Missing claims")
    except Exception:
        raw2, t2, s2 = chat_generate(model, tokenizer, prompt + "\nReturn valid JSON only.", CLAIM_CHECK_TOKEN_LIMIT)
        tokens += t2; seconds += s2
        parsed = parse_json_object(raw2)
    allowed = {"supported", "contradicted", "not_found", "unclear"}
    norm_src = normalize_quote(source_used)
    sum_nums = set(numeric_tokens(summary))
    results = []
    for item in parsed.get("claims", [])[:12]:
        claim = str(item.get("claim", "")).strip()
        status = str(item.get("status", "unclear")).strip().lower()
        evidence = str(item.get("evidence", "")).strip()
        if not claim: continue
        if status not in allowed: status = "unclear"
        extra_nums = sorted(set(numeric_tokens(claim)) - sum_nums)
        q_ok = bool(evidence) and normalize_quote(evidence) in norm_src
        if extra_nums: status, evidence, q_ok = "checker_mismatch", "", False
        elif status in {"supported", "contradicted"} and not evidence: status = "no_evidence_found"
        elif status in {"supported", "contradicted"} and not q_ok: status, evidence = "evidence_not_verified", ""
        if status in {"not_found", "unclear", "no_evidence_found"}: evidence, q_ok = "", False
        results.append({"claim": claim, "assessment": status, "source_evidence": evidence, "quote_found_in_model_input": bool(q_ok)})
    return results, tokens, seconds

def evaluate_one(source_text, model_key, prompt_style, archetype, model, tokenizer):
    source_used, orig_toks, truncated = prepare_source(source_text, tokenizer)
    anchors = extract_fact_anchors(source_used, archetype=archetype)
    summary, sum_toks, sum_sec, cap_hit = make_summary(source_used, prompt_style, model, tokenizer, fact_anchors=anchors, archetype=archetype)
    _, init_date_corr = correct_unique_source_date(source_used, summary)
    _, init_name_corr = correct_source_name_spellings(source_used, summary)
    if unsupported_numeric_tokens(source_used, summary) or unsupported_full_dates(source_used, summary) or init_date_corr or init_name_corr:
        try:
            r_sum, r_toks, r_sec, r_cap = make_summary(source_used, prompt_style, model, tokenizer, numeric_retry=True, fact_anchors=anchors, archetype=archetype)
            if r_sum:
                summary, sum_toks, sum_sec, cap_hit = r_sum, sum_toks + r_toks, sum_sec + r_sec, r_cap
        except Exception:
            pass
    summary, name_corrs = correct_source_name_spellings(source_used, summary)
    summary, date_corr = correct_unique_source_date(source_used, summary)
    num_mis = unsupported_numeric_tokens(source_used, summary)
    date_mis = unsupported_full_dates(source_used, summary)
    claim_err = None
    try:
        claims, chk_toks, chk_sec = check_claims(source_used, summary, model, tokenizer)
    except Exception as e:
        claims, chk_toks, chk_sec, claim_err = [], 0, 0.0, str(e)
    for c in claims:
        if c["assessment"] == "checker_mismatch":
            c["independent_nli_label"], c["independent_nli_score"] = "skipped", None
        else:
            prem = c["source_evidence"] if c["quote_found_in_model_input"] else source_used
            c["independent_nli_label"], c["independent_nli_score"] = nli_check_claim(prem, c["claim"], archetype=archetype)
    needs_review = bool(
        claim_err or num_mis or date_mis or date_corr or name_corrs or not claims
        or any(c["assessment"] != "supported" or not c["quote_found_in_model_input"] or c["independent_nli_label"] == "contradiction" for c in claims)
    )
    if not needs_review:
        outcome = "supported"
    elif num_mis or date_mis or claim_err or not claims or any(c["assessment"] in {"contradicted", "evidence_not_verified", "checker_mismatch"} or c["independent_nli_label"] == "contradiction" for c in claims):
        outcome = "major"
    else:
        outcome = "minor"
    src_words = max(len(source_used.split()), 1)
    sum_words = len(summary.split())
    return {
        "archetype": archetype, "model": model_key, "prompt": prompt_style,
        "summary": summary, "outcome": outcome, "needs_review": needs_review,
        "summary_words": sum_words, "compression": round(sum_words / src_words, 4),
        "gen_seconds": round(sum_sec, 3), "check_seconds": round(chk_sec, 3),
        "n_claims": len(claims),
        "n_supported_claims": sum(1 for c in claims if c["assessment"] == "supported"),
        "n_contradictions": sum(1 for c in claims if c.get("independent_nli_label") == "contradiction"),
    }

## 2. Load Both 30-Article Samples (Roman-Script Hinglish & Devanagari Code-Mixed)

In [ ]:
PARQUET_URL = "https://huggingface.co/datasets/suyash2739/News_Hinglish_English/resolve/22a19b81fd8cc151a9d011cfdd7e6dd4e9f2b359/data/train-00000-of-00001.parquet"
df_rom = pd.read_parquet(io.BytesIO(requests.get(PARQUET_URL, timeout=120).content))
DEV_RE = re.compile(r"[\u0900-\u097F]")
LATIN_RE = re.compile(r"[A-Za-z]")
df_rom["content"] = df_rom["content"].map(lambda x: re.sub(r"\s+", " ", str(x)).strip())
df_rom["Hinglish"] = df_rom["Hinglish"].map(lambda x: re.sub(r"\s+", " ", str(x)).strip())
eligible_rom = df_rom.index[
    (df_rom["content"] != "") & (df_rom["Hinglish"] != "") &
    (~df_rom["Hinglish"].map(lambda x: bool(DEV_RE.search(x)))) &
    df_rom["Hinglish"].map(lambda x: bool(LATIN_RE.search(x))) &
    (~df_rom.duplicated(subset=["content", "Hinglish"], keep="first"))
].tolist()
roman_indices = sorted(random.Random(SEED).sample(eligible_rom, N_SAMPLE))
roman_samples = [{"source_id": f"rom_{int(i):05d}", "archetype": "roman_hinglish", "source": df_rom.at[i, "Hinglish"]} for i in roman_indices]

xlsum_hi = load_dataset("csebuetnlp/xlsum", "hindi", split="test", revision="ab04842511fc357feae53281fd104aa3c36dff07")
dev_indices = sorted(random.Random(SEED).sample(range(len(xlsum_hi)), N_SAMPLE))
devanagari_samples = []
for idx in dev_indices:
    mixed_text = code_mix_text(xlsum_hi[int(idx)]["text"], p=1.0, seed=SEED + int(idx))
    devanagari_samples.append({"source_id": f"dev_{int(idx):05d}", "archetype": "devanagari_codemixed", "source": mixed_text})

all_samples = roman_samples + devanagari_samples
print(f"Loaded {len(roman_samples)} Roman-script + {len(devanagari_samples)} Devanagari/code-mixed = {len(all_samples)} sources.")

## 3. Run Dual-Archetype Re-Evaluation (240 Outputs Total)

In [ ]:
OUT_JSONL = Path("/content/factindic_dual_reeval_rows.jsonl")
completed = {}
if OUT_JSONL.exists():
    for line in OUT_JSONL.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rec = json.loads(line)
            completed[(rec["source_id"], rec["model"], rec["prompt"])] = rec

total_tasks = len(all_samples) * len(MODEL_IDS) * len(PROMPT_STYLES)
done_count = len(completed)

def render_ring(pct, msg):
    r = 32; c = 2 * 3.14159265 * r; off = c * (1 - pct / 100.0)
    return (
        f'<div style="display:flex;align-items:center;gap:14px">'
        f'<svg width="80" height="80" viewBox="0 0 80 80"><circle cx="40" cy="40" r="{r}" fill="none" stroke="#e2e8f0" stroke-width="7"/>'
        f'<circle cx="40" cy="40" r="{r}" fill="none" stroke="#0d9488" stroke-width="7" stroke-linecap="round" stroke-dasharray="{c:.2f}" stroke-dashoffset="{off:.2f}" transform="rotate(-90 40 40)"/>'
        f'<text x="40" y="45" text-anchor="middle" font-family="sans-serif" font-size="15" font-weight="700" fill="#0d9488">{pct}%</text></svg>'
        f'<div style="font-family:sans-serif;font-size:13px;font-weight:600;color:#0f172a">{html.escape(msg)}</div></div>'
    )

status_widget = widgets.HTML(value=render_ring(int(100 * done_count / max(total_tasks, 1)), f"Starting ({done_count}/{total_tasks} cached)…"))
display(status_widget)

with OUT_JSONL.open("a", encoding="utf-8") as fout:
    for model_key in MODEL_IDS:
        model, tokenizer = load_model(model_key)
        for prompt_style in PROMPT_STYLES:
            for item in all_samples:
                key = (item["source_id"], model_key, prompt_style)
                if key in completed:
                    continue
                res = evaluate_one(item["source"], model_key, prompt_style, item["archetype"], model, tokenizer)
                res["source_id"] = item["source_id"]
                fout.write(json.dumps(res, ensure_ascii=False) + "\n")
                fout.flush()
                completed[key] = res
                done_count += 1
                pct = int(100 * done_count / total_tasks)
                status_widget.value = render_ring(pct, f"{done_count}/{total_tasks} · {model_key} · {prompt_style} · {item['archetype']}")

print("Finished all", len(completed), "evaluations.")

## 4. Aggregate Scores, 95% Paired Bootstrap CIs & Export

In [ ]:
rows = list(completed.values())
df_res = pd.DataFrame(rows)
df_res.to_csv("/content/factindic_dual_archetype_reeval_rows.csv", index=False)

def bootstrap_diff_ci(a_vals, b_vals, n_boot=5000, seed=42):
    rng = random.Random(seed)
    n = len(a_vals)
    diffs = [a - b for a, b in zip(a_vals, b_vals)]
    obs = sum(diffs) / n * 100.0
    boots = sorted(sum(diffs[rng.randrange(n)] for _ in range(n)) / n * 100.0 for _ in range(n_boot))
    return round(obs, 2), round(boots[int(0.025 * n_boot)], 2), round(boots[int(0.975 * n_boot)], 2)

summary_rows = []
for arch in ["roman_hinglish", "devanagari_codemixed"]:
    for model_key in MODEL_IDS:
        for prompt_style in PROMPT_STYLES:
            sub = df_res[(df_res["archetype"] == arch) & (df_res["model"] == model_key) & (df_res["prompt"] == prompt_style)]
            cnt = Counter(sub["outcome"])
            n = len(sub)
            err_rate = round((cnt["minor"] + cnt["major"]) / max(n, 1) * 100.0, 1)
            summary_rows.append({
                "archetype": arch, "model": model_key, "prompt": prompt_style, "n": n,
                "supported": cnt["supported"], "minor": cnt["minor"], "major": cnt["major"],
                "error_rate_pct": err_rate, "pass_rate_pct": round(100.0 - err_rate, 1),
                "mean_words": round(sub["summary_words"].mean(), 2),
                "mean_compression": round(sub["compression"].mean(), 4),
                "mean_gen_sec": round(sub["gen_seconds"].mean(), 3),
            })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

paired_diffs = []
for arch in ["roman_hinglish", "devanagari_codemixed"]:
    for model_key in MODEL_IDS:
        std = df_res[(df_res["archetype"] == arch) & (df_res["model"] == model_key) & (df_res["prompt"] == "Standard")].sort_values("source_id")
        fact = df_res[(df_res["archetype"] == arch) & (df_res["model"] == model_key) & (df_res["prompt"] == "Factuality-aware")].sort_values("source_id")
        obs, lo, hi = bootstrap_diff_ci([1 if x != "supported" else 0 for x in fact["outcome"]], [1 if x != "supported" else 0 for x in std["outcome"]])
        paired_diffs.append({"archetype": arch, "model": model_key, "diff_pp": obs, "ci_low": lo, "ci_high": hi})

out_summary = {"summary_table": summary_rows, "paired_prompt_diffs": paired_diffs}
Path("/content/factindic_dual_archetype_reeval_summary.json").write_text(json.dumps(out_summary, indent=2), encoding="utf-8")
print("Paired prompt diffs (pp):", paired_diffs)
files.download("/content/factindic_dual_archetype_reeval_summary.json")
files.download("/content/factindic_dual_archetype_reeval_rows.csv")